[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [46]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.rank= rank
        self.linear= nn.Linear(in_features, out_features)
        self.lora_A= nn.Parameter(torch.randn(self.rank, in_features))
        self.lora_B= nn.Parameter(torch.zeros(out_features, self.rank))
        self.alpha= alpha
        for param in self.linear.parameters():
            param.requires_grad = False
    def forward(self, x):
      frozen_h= self.linear(x).detach()
      lora_h= (self.alpha/self.rank)*torch.matmul(torch.matmul(self.lora_B, self.lora_A), x.T).T
      return frozen_h + lora_h

In [47]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

frozen_h: torch.Size([2, 8])
lora_h: torch.Size([2, 8])
Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [48]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (1.7ms)
  ✅ [2/5] LoRA parameter shapes (1.2ms)
frozen_h: torch.Size([2, 4])
lora_h: torch.Size([2, 4])
  ✅ [3/5] B=0 means output equals base (6.3ms)
frozen_h: torch.Size([2, 4])
lora_h: torch.Size([2, 4])
  ✅ [4/5] Only LoRA params get gradients (2.4ms)
frozen_h: torch.Size([3, 4])
lora_h: torch.Size([3, 4])
  ✅ [5/5] Forward computation (2.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (14.5ms total)
  Progress saved. Run status() to see your dashboard.

